# 🧪 Container Recognition: Custom LSTM-CNN Mode

This notebook demonstrates the **Research Path** model for shipping container recognition. 

### **The Architecture**
1. **Detection**: YOLOv8 locates the alphanumeric panel.
2. **Recognition**: A custom **Convolutional Recurrent Neural Network (CRNN)** performs the reading. This model uses **Bi-LSTM** layers to understand the sequence of characters.
3. **Correction**: The raw output is passed through our **ISO 6346 Smart Corrector** to mathematically validate the check digit.

> **Benefit**: This model is 10x faster and 100x smaller than general OCR libraries.

In [ ]:
import os
import cv2
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Ensure project root is in path
sys.path.append(os.getcwd())

from src.inference.pipeline import load_models, read_crnn
from src.utils.formatters import format_container_code
from src.config import CONTAINER_TEST_DATA

## 1. Load the Custom Expert Model
We load the YOLO detector and the custom **LSTM-CNN** weights.

In [ ]:
os.environ['DETECT_MODE'] = 'container'
os.environ['OCR_TYPE'] = 'expert'
device = "cuda" if torch.cuda.is_available() else "cpu"

yolo_model, ocr_engine, charset = load_models(device, mode='container')
print(f"✓ LSTM-CNN Expert Model loaded on {device}")

## 2. Interactive Testing
Select an image to see the custom model in action.

In [ ]:
# Select a random image from the test set
test_images = list(CONTAINER_TEST_DATA.glob('*.jpg'))
if not test_images:
    print("No test images found.")
else:
    img_path = str(test_images[0])
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Stage 1: Detection
    results = yolo_model(img, verbose=False)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.title("Custom Model Analysis")
    plt.axis('off')
    plt.show()
    
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            pad = 15
            crop = img[max(0, y1-pad):min(img.shape[0], y2+pad), max(0, x1-pad):min(img.shape[1], x2+pad)]
            
            # Stage 2: LSTM-CNN Inference
            raw_text = read_crnn(crop, ocr_engine, charset, device)
            
            # Stage 3: ISO Smart Correction
            corrected_text = format_container_code(raw_text)
            
            print(f"--- LSTM-CNN Result ---")
            print(f"Sequence Output:   {raw_text}")
            print(f"ISO Validated:     {corrected_text}")